# MEV — Máxima Extração de Valor na Rede Polygon
**Pesquisa FAPEMIG** | Coordenador: Prof. José Augusto Miranda Nacif | Bolsista: Aline Cristina Santos Silva

---

Este notebook documenta a **coleta e análise de dados on-chain** da rede Polygon (Layer-2),
com foco na medição do *Reordering Slippage* em swaps da Uniswap V3.

### Objetivo
Coletar eventos de Swap reais da blockchain para, posteriormente, calcular o slippage de reordenação
e comparar com a linha de base da Ethereum Mainnet — validando a hipótese **H** da proposta:
> *Contratos inteligentes em redes L2 apresentam um reordering slippage significativamente menor do que na rede principal Ethereum.*

### Perguntas de Pesquisa endereçadas
- **RQ1:** Qual a magnitude da diferença no Reordering Slippage médio entre Ethereum Mainnet e Polygon?
- **RQ2:** Qual a proporção entre Slippage Adversário (MEV) e Slippage de Colisão (benigno)?
- **RQ3:** Ativos voláteis (memecoins) apresentam maior slippage adversário também em L2?

---

## 1. Instalação de Dependências

Bibliotecas necessárias:
- **`web3`** — interface com nós da blockchain via RPC
- **`pandas`** — manipulação e análise dos dados coletados
- **`python-dotenv`** — carrega variáveis de ambiente do arquivo `.env` (protege a API key)
- **`matplotlib` / `seaborn`** — visualizações

In [2]:
# Execute uma vez para instalar as dependências
%pip install web3 pandas python-dotenv matplotlib seaborn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuração — Carregando Credenciais

A API key da Alchemy fica armazenada no arquivo **`.env`** (nunca versionado no GitHub).
O arquivo `.env.example` no repositório documenta quais variáveis são necessárias sem expor os valores reais.



## 3. Conexão com a Rede Polygon via Alchemy

A conexão é feita via **RPC (Remote Procedure Call)** — um protocolo que permite consultar
o estado da blockchain sem precisar baixar todos os dados localmente.

A **Alchemy** atua como provedor de nó, fornecendo acesso à Polygon Mainnet de forma confiável e escalável.

**Por que Polygon?**
Por ser uma rede L2, espera-se que o *reordering slippage* seja menor do que na Ethereum Mainnet —
essa é exatamente a hipótese que esta pesquisa busca validar empiricamente.

In [3]:
import os
import requests
from dotenv import load_dotenv
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware

load_dotenv(override=True)
API_KEY = os.getenv("ALCHEMY_API_KEY")
RPC_URL = f"https://polygon-mainnet.g.alchemy.com/v2/{API_KEY}"

# Sessão customizada sem verificação SSL
session = requests.Session()
session.verify = False

from web3.middleware import ExtraDataToPOAMiddleware
from requests.adapters import HTTPAdapter

w3 = Web3(Web3.HTTPProvider(RPC_URL, session=session))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)

try:
    bloco = w3.eth.block_number
    print(f" Conectado! Bloco: {bloco:,}")
except Exception as e:
    print(f" Erro: {e}")

 Conectado! Bloco: 86,319,932


\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 4. Definição do Contrato — Pool Uniswap V3 (USDC/WETH)

Na Uniswap V3, cada par de tokens possui um **contrato de pool dedicado**.
Toda vez que um swap ocorre, o contrato emite um **evento `Swap`** — um log público e imutável
gravado na blockchain, contendo:

| Campo | Descrição | Relevância para a pesquisa |
|---|---|---|
| `sqrtPriceX96` | Preço de execução real codificado | Base para calcular o slippage |
| `amount0 / amount1` | Volume negociado de cada token | Tamanho do swap (impacto no preço) |
| `sender / recipient` | Endereços envolvidos | Identificar padrões de ataque sandwich |
| `tick` | Posição na curva de preço | Faixa de liquidez utilizada |
| `blockNumber` | Bloco em que ocorreu | Agrupar transações por bloco para cálculo do reordering slippage |

Começamos com o pool **USDC/WETH 0.05%** por ser um dos mais líquidos na Polygon —
ideal para calibrar a metodologia antes de analisar ativos voláteis (RQ3).

In [4]:
# Endereço do pool USDC/WETH 0.05% na Polygon (Uniswap V3)
POOL_ADDRESS = Web3.to_checksum_address("0x45dda9cb7c25131df268515131f647d726f50608")

# ABI mínimo — apenas o evento Swap (não precisamos do ABI completo)
POOL_ABI = [
    {
        "anonymous": False,
        "inputs": [
            {"indexed": True,  "name": "sender",       "type": "address"},
            {"indexed": True,  "name": "recipient",    "type": "address"},
            {"indexed": False, "name": "amount0",      "type": "int256"},
            {"indexed": False, "name": "amount1",      "type": "int256"},
            {"indexed": False, "name": "sqrtPriceX96", "type": "uint160"},
            {"indexed": False, "name": "liquidity",    "type": "uint128"},
            {"indexed": False, "name": "tick",         "type": "int24"}
        ],
        "name": "Swap",
        "type": "event"
    }
]

contrato = w3.eth.contract(address=POOL_ADDRESS, abi=POOL_ABI)
print(f" Contrato carregado: {POOL_ADDRESS}")
print(f" Pool: USDC/WETH 0.05% — Uniswap V3 na Polygon")

 Contrato carregado: 0x45dDa9cb7c25131DF268515131f647d726f50608
 Pool: USDC/WETH 0.05% — Uniswap V3 na Polygon


## 5. Coleta de Eventos de Swap

Consultamos os **logs da blockchain** em um intervalo de blocos recentes,
filtrando apenas os eventos `Swap` do pool selecionado.

Na Polygon, cada bloco leva ~2 segundos — 500 blocos ≈ últimos 16 minutos de transações.

> **Nota metodológica:** Para a análise completa do Reordering Slippage, será necessário
> coletar um volume maior de blocos. Esta célula serve como validação da pipeline de coleta.

In [8]:
import time

JANELA_BLOCOS = 50

bloco_fim    = w3.eth.block_number
bloco_inicio = bloco_fim - 200

print(f" Coletando eventos Swap em lotes de {JANELA_BLOCOS} blocos...")
print(f"   Bloco inicial : {bloco_inicio:,}")
print(f"   Bloco final   : {bloco_fim:,}")
print()

todos_eventos = []

for inicio in range(bloco_inicio, bloco_fim, JANELA_BLOCOS):
    fim = min(inicio + JANELA_BLOCOS - 1, bloco_fim)
    try:
        eventos = contrato.events.Swap.get_logs(
            from_block=inicio,
            to_block=fim
        )
        todos_eventos.extend(eventos)
        print(f"    Blocos {inicio:,} → {fim:,} | {len(eventos)} swaps")
    except Exception as e:
        print(f"    Blocos {inicio:,} → {fim:,} | Erro: {e}")
    time.sleep(0.3)

print(f"\n Total coletado: {len(todos_eventos)} swaps")
eventos = todos_eventos

\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


 Coletando eventos Swap em lotes de 50 blocos...
   Bloco inicial : 86,319,797
   Bloco final   : 86,319,997



\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://

    Blocos 86,319,797 → 86,319,846 | Erro: 400 Client Error: Bad Request for url: https://polygon-mainnet.g.alchemy.com/v2/H2W733DJauMo34FthZ26E


\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://

    Blocos 86,319,847 → 86,319,896 | Erro: 400 Client Error: Bad Request for url: https://polygon-mainnet.g.alchemy.com/v2/H2W733DJauMo34FthZ26E


\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://

    Blocos 86,319,897 → 86,319,946 | Erro: 400 Client Error: Bad Request for url: https://polygon-mainnet.g.alchemy.com/v2/H2W733DJauMo34FthZ26E


\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
\\wsl.localhost\Ubuntu\home\aline\slippage-analysis\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://

    Blocos 86,319,947 → 86,319,996 | Erro: 400 Client Error: Bad Request for url: https://polygon-mainnet.g.alchemy.com/v2/H2W733DJauMo34FthZ26E

 Total coletado: 0 swaps
